# Question Answering with Web Search
This notebook builds a lightweight QA workflow for general-knowledge questions using spaCy for parsing, Wikipedia for retrieval, and simple answer extraction heuristics.


## Example Questions
The notebook tests the pipeline on a mix of factual questions such as capitals, authors, dates, locations, and counts.


## Workflow
- Extract noun phrases from each question with spaCy
- Retrieve relevant Wikipedia pages
- Extract concise answers from the retrieved text


# Question Parsing
Use spaCy noun chunks to identify key phrases in each question.


# Web Retrieval
Use Wikipedia search results and page text as the retrieval layer.


# Answer Extraction
Use text processing and lightweight NLP heuristics to extract answers from retrieved content.


In [ ]:
!pip install spacy requests beautifulsoup4 transformers torch
!python -m spacy download en_core_web_sm
import spacy
import requests
from bs4 import BeautifulSoup
from transformers import pipeline

In [ ]:
nlp = spacy.load("en_core_web_sm")

questions = [
    "What is the capital of France?",
    "Who wrote Hamlet?",
    "When was Albert Einstein born?",
    "What is the largest desert in the world?",
    "Where is Mount Everest located?",
    "What are the primary colors?",
    "Who was the President of the United States in 1945?",
    "What is the population of Tokyo in 2023?",
    "Which country has the longest coastline in the world?",
    "How many moons does Jupiter have, and what are their names?"
]

def extract_noun_phrases(question: str):
    doc = nlp(question)
    noun_chunks = [chunk.text.strip() for chunk in doc.noun_chunks]
    return list(dict.fromkeys(noun_chunks))

for question in questions:
    print("QUESTION:", question)
    print("NOUN PHRASES:", extract_noun_phrases(question))
    print()

In [ ]:
WIKI_SEARCH_URL = "https://en.wikipedia.org/w/api.php"

def wikipedia_search(query: str):
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "utf8": "",
        "format": "json",
    }
    headers = {"User-Agent": "Mozilla/5.0 (compatible; wiki-qa-notebook/1.0)"}
    response = requests.get(WIKI_SEARCH_URL, params=params, headers=headers, timeout=10)
    response.raise_for_status()
    data = response.json()

    if not data["query"]["search"]:
        return None

    title = data["query"]["search"][0]["title"]
    return "https://en.wikipedia.org/wiki/" + title.replace(" ", "_")

In [ ]:
def fetch_page_text(url: str, max_paragraphs: int = 8) -> str:
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    paragraphs = []

    for p in soup.select("p"):
        text = p.get_text(" ", strip=True)
        if text:
            paragraphs.append(text)
        if len(paragraphs) >= max_paragraphs:
            break

    return "\n".join(paragraphs)

In [ ]:
def split_into_sentences(text: str) -> list[str]:
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

def find_supporting_sentence(context: str, answer: str) -> str:
    if not context.strip():
        return ""

    sentences = split_into_sentences(context)
    answer_lower = answer.lower().strip()

    if answer_lower:
        for sentence in sentences:
            if answer_lower in sentence.lower():
                return sentence

    return sentences[0] if sentences else ""

In [ ]:

qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)

def qa_model_answer(question: str, context: str) -> str:
    if not context.strip():
        return ""
    result = qa_pipeline(question=question, context=context)
    return result.get("answer", "").strip()

In [ ]:
def get_context_for_question(question: str, noun_phrases: list[str]) -> str:
    query = question
    if noun_phrases:
        query += " " + noun_phrases[0]

    wiki_url = wikipedia_search(query)
    if not wiki_url:
        return ""

    return fetch_page_text(wiki_url)

In [ ]:
def answer_question(question: str):
    noun_phrases = extract_noun_phrases(question)
    context = get_context_for_question(question, noun_phrases)
    answer = qa_model_answer(question, context)
    supporting_sentence = find_supporting_sentence(context, answer)

    return {
        "question": question,
        "noun_phrases": noun_phrases,
        "context_excerpt": context[:300] + "...",
        "answer": answer,
        "supporting_sentence": supporting_sentence,
    }

for q in questions:
    result = answer_question(q)
    print("====================================================")
    print("QUESTION:", result["question"])
    print("NOUN PHRASES:", result["noun_phrases"])
    print("CONTEXT EXCERPT:", result["context_excerpt"])
    print("ANSWER:", result["answer"])
    print("SUPPORTING SENTENCE:", result["supporting_sentence"])
    print()